# U07 · Series temporales — ingresos diarios en urgencias

> **El tiempo como dato.** Hasta ahora cada fila era un paciente o un centro, independiente de los demás: podíamos barajarlos sin perder nada. Aquí eso cambia por completo. Trabajamos con `urgencias_diarias.csv`, una serie: los ingresos de **hoy** están relacionados con los de **ayer**, con los del **lunes pasado** y con los del **invierno anterior**. El orden de las filas es, en sí mismo, información — y si lo rompemos, lo perdemos.

> ⚠️ Todos los datos son **sintéticos** (generados por código, semilla fija). **No representan un hospital real.** La primera celda los crea sola: no hay que descargar nada.

**Qué haremos, en clave clínica:**
1. **Cargar y dibujar** la serie de ingresos diarios en urgencias, e interpretar lo que se ve a simple vista.
2. **Estacionalidad**: ¿qué día de la semana y qué mes cargan más? Lo veremos en barras.
3. La lección **crítica** de la unidad: por qué un reparto al azar entre entrenamiento y test es una **trampa** en una serie temporal, y cómo hacer bien el **corte temporal**.
4. Un **modelo de Machine Learning con features de calendario** (día de la semana, mes, festivo, temporada de gripe, temperatura) para predecir los ingresos del futuro, evaluado *solo* sobre ese futuro.
5. Comparar el modelo con un **baseline ingenuo** — ¿de verdad aporta algo más que "mirar lo que pasó la semana pasada"?

[Abrir en Colab](PENDIENTE_DRIVE)


## 0. Preparamos los datos (esta celda se ejecuta sola)

La primera celda de cada cuaderno del curso **genera los ficheros sintéticos** en la carpeta de trabajo, incluido `urgencias_diarias.csv`. Ejecútala una vez (▶ o *Mayús+Enter*) y sigue hacia abajo. Recuerda: son datos **inventados**, sirven para aprender, no para decidir sobre un hospital real.


In [ ]:
# === Generación de los datos sintéticos del curso (se ejecuta sola) ===
# TODOS LOS DATOS SON SINTÉTICOS. No representan pacientes reales. Semilla fija -> reproducible.
import os, numpy as np, pandas as pd

def generar_datos_clinicos(carpeta="."):
    """Crea los CSV del curso si no están ya en la carpeta. Devuelve la ruta."""
    if os.path.exists(os.path.join(carpeta, "pacientes.csv")):
        return carpeta
    RNG = np.random.default_rng(2026)   # misma semilla en todo el curso

    # --- Centros ---
    AREAS = ["Valencia","Alicante","Castellón","Madrid","Barcelona","Sevilla","Bilbao","Zaragoza"]
    nc = 24
    tipo = RNG.choice(["hospital","centro de salud"], nc, p=[0.35,0.65])
    camas = np.where(tipo=="hospital", RNG.integers(120,900,nc), RNG.integers(0,25,nc))
    nserv = np.where(tipo=="hospital", RNG.integers(12,40,nc), RNG.integers(3,10,nc))
    centros = pd.DataFrame({"centro_id":[f"C{i:03d}" for i in range(1,nc+1)],"tipo":tipo,
        "area":RNG.choice(AREAS,nc),"camas":camas,"n_servicios":nserv,
        "urgencias_dia_media":np.where(tipo=="hospital",RNG.integers(80,320,nc),RNG.integers(5,40,nc)),
        "ratio_mayores65":RNG.uniform(0.12,0.34,nc).round(3)})
    centros.to_csv(os.path.join(carpeta,"centros.csv"), index=False)

    # --- Pacientes (tabla central) ---
    N = 20000
    sexo = RNG.choice(["M","F"], N, p=[0.49,0.51])
    edad = np.clip(np.where(RNG.random(N)<0.6, RNG.normal(58,15,N), RNG.normal(40,12,N)),18,95).round().astype(int)
    p_act = np.clip(0.28-0.0016*(edad-18),0.06,0.28); p_ex = np.clip(0.10+0.004*(edad-18),0.10,0.40); p_nu = 1-p_act-p_ex
    tabaquismo = np.array([RNG.choice(["nunca","ex","activo"],p=[a,b,c]) for a,b,c in zip(p_nu,p_ex,p_act)])
    actividad = RNG.choice(["baja","media","alta"], N, p=[0.4,0.4,0.2])
    antec = RNG.binomial(1,0.22,N)
    actn = pd.Series(actividad).map({"baja":1.5,"media":0.0,"alta":-1.3}).values
    imc = np.clip(RNG.normal(26.5,4.2,N)+0.03*(edad-50)+actn+RNG.normal(0,1.0,N),15,48).round(1)
    fa = (tabaquismo=="activo").astype(float); fe = (tabaquismo=="ex").astype(float)
    ta_s = np.clip(RNG.normal(120,12,N)+0.45*(edad-45)+0.7*(imc-25)+6*fa+RNG.normal(0,6,N),85,220).round().astype(int)
    ta_d = np.clip(0.55*ta_s+RNG.normal(20,6,N),50,130).round().astype(int)
    glu = np.clip(RNG.normal(100,12,N)+1.0*(imc-25)+0.3*(edad-45)+9*antec+RNG.normal(0,7,N),60,320).round(1)
    diabetes = (glu>126).astype(int)
    col = np.clip(RNG.normal(195,30,N)+0.4*(edad-45)+1.1*(imc-25)+RNG.normal(0,15,N),110,380).round(1)
    hdl = np.clip(RNG.normal(55,12,N)-0.25*(imc-25)+6*(sexo=="F")-5*fa-3*actn+RNG.normal(0,6,N),20,110).round(1)
    z = (-3.1+0.062*(edad-50)+0.019*(ta_s-120)+0.008*(col-190)-0.028*(hdl-55)+0.055*(imc-25)
         +0.85*fa+0.30*fe+0.60*diabetes+0.45*antec+0.55*(sexo=="M")
         +0.012*np.maximum(edad-65,0)*fa+RNG.normal(0,0.35,N))
    p = 1/(1+np.exp(-z))
    riesgo = (100*p).round(1); evento = RNG.binomial(1,p)
    pacientes = pd.DataFrame({"paciente_id":[f"P{i:05d}" for i in range(1,N+1)],"edad":edad,"sexo":sexo,
        "imc":imc,"ta_sistolica":ta_s,"ta_diastolica":ta_d,"glucemia":glu,"colesterol_total":col,"hdl":hdl,
        "tabaquismo":tabaquismo,"actividad_fisica":actividad,"antecedentes_familiares":antec,
        "diabetes":diabetes,"riesgo_cv_10a":riesgo,"evento_cv":evento})
    pacientes.to_csv(os.path.join(carpeta,"pacientes.csv"), index=False)

    # --- Versión "sucia" para EDA ---
    d = pacientes.copy()
    d["sexo"] = d["sexo"].map(lambda s: RNG.choice({"M":["M","m","Masculino","H","Hombre"],"F":["F","f","Femenino","Mujer"]}[s]))
    idx = RNG.choice(d.index,int(N*0.06),replace=False)
    d["glucemia"] = d["glucemia"].astype(object)   # permite mezclar texto (mmol/L con coma) y números
    d.loc[idx,"glucemia"] = [str(round(v/18.0,1)).replace(".",",") for v in d.loc[idx,"glucemia"]]
    prob_na = np.where(pacientes["edad"]<40,0.12,0.04); mask = RNG.random(N)<prob_na
    d.loc[mask,"hdl"] = np.nan
    d.loc[RNG.choice(d.index,int(N*0.05),replace=False),"colesterol_total"] = np.nan
    d.loc[RNG.choice(d.index,25,replace=False),"edad"] = RNG.integers(150,260,25)
    d.loc[RNG.choice(d.index,20,replace=False),"ta_sistolica"] = -RNG.integers(1,90,20)
    d.loc[RNG.choice(d.index,15,replace=False),"imc"] = RNG.uniform(80,200,15).round(1)
    d["colesterol_total"] = d["colesterol_total"].astype(object); d["ta_diastolica"] = d["ta_diastolica"].astype(object)
    d.loc[RNG.choice(d.index,30,replace=False),"colesterol_total"] = "desconocido"
    d.loc[RNG.choice(d.index,20,replace=False),"ta_diastolica"] = "N/D"
    d["tabaquismo"] = d["tabaquismo"].map(lambda s: RNG.choice({"nunca":["nunca","No fumador","NUNCA"],
        "ex":["ex","exfumador","Ex-fumador"],"activo":["activo","Fumador","SI"]}[s]))
    d = pd.concat([d, d.sample(400,random_state=3)], ignore_index=True)
    idx = RNG.choice(d.index,200,replace=False); d.loc[idx,"paciente_id"] = d.loc[idx,"paciente_id"].map(lambda s:" "+s+" ")
    d = d.sample(frac=1,random_state=11).reset_index(drop=True)
    d.to_csv(os.path.join(carpeta,"pacientes_sucio.csv"), index=False)

    # --- Urgencias (serie temporal) ---
    ND = 730; fechas = pd.date_range("2024-01-01", periods=ND, freq="D")
    doy = fechas.dayofyear.values; dow = fechas.dayofweek.values
    fest_set = {(1,1),(1,6),(5,1),(8,15),(10,12),(11,1),(12,6),(12,8),(12,25)}
    festivo = np.array([1 if (f.month,f.day) in fest_set else 0 for f in fechas])
    gripe = np.array([1 if f.month in (12,1,2) else 0 for f in fechas])
    temp = (15+10*np.sin(2*np.pi*(doy-110)/365)+RNG.normal(0,2.5,ND)).round(1)
    est = 1+0.14*np.sin(2*np.pi*(doy-20)/365)
    sem = np.where(dow==0,1.18,np.where(dow>=5,1.10,1.0))
    mu = 120.0*est*sem*(1+0.22*gripe)*(1+0.15*festivo)
    ingresos = RNG.poisson(np.maximum(mu,1)).astype(int)
    pd.DataFrame({"fecha":fechas.strftime("%Y-%m-%d"),"ingresos":ingresos,"festivo":festivo,
        "temporada_gripe":gripe,"temperatura":temp}).to_csv(os.path.join(carpeta,"urgencias_diarias.csv"), index=False)

    # --- Notas clínicas (texto) ---
    plant = {"cardiología":["dolor torácico opresivo de {t} de evolución, irradiado a brazo izquierdo",
        "disnea de esfuerzo progresiva y palpitaciones, antecedente de hipertensión",
        "control de anticoagulación, fibrilación auricular conocida, sin dolor actual",
        "edemas en miembros inferiores y ortopnea, sospecha de insuficiencia cardiaca"],
      "respiratorio":["tos productiva de {t}, fiebre y disnea, auscultación con crepitantes",
        "sibilancias y opresión torácica, antecedente de asma, mala respuesta a inhalador",
        "disnea súbita y dolor pleurítico, saturación disminuida",
        "tos seca persistente y febrícula, contacto con caso respiratorio"],
      "digestivo":["dolor abdominal en fosa iliaca derecha de {t}, náuseas y febrícula",
        "epigastralgia y pirosis, relación con las comidas, sin signos de alarma",
        "diarrea de {t} sin productos patológicos, buen estado general",
        "ictericia y coluria, dolor en hipocondrio derecho"],
      "neurología":["cefalea intensa de inicio súbito, la peor de su vida, con fotofobia",
        "focalidad neurológica de {t}, debilidad en hemicuerpo y disartria",
        "mareo con giro de objetos y vómitos, sin cortejo vegetativo",
        "crisis comicial presenciada, recuperación progresiva del nivel de conciencia"],
      "traumatología":["caída casual con dolor e impotencia funcional en muñeca, deformidad",
        "lumbalgia mecánica de {t} tras esfuerzo, sin déficit neurológico",
        "esguince de tobillo con edema y dificultad para la deambulación",
        "gonalgia y derrame articular tras traumatismo deportivo"]}
    prio = {"cardiología":[0.35,0.45,0.20],"respiratorio":[0.30,0.45,0.25],"digestivo":[0.20,0.50,0.30],
        "neurología":[0.45,0.35,0.20],"traumatología":[0.12,0.48,0.40]}
    tiempos = ["horas","un día","dos días","una semana","varios días"]; esp_list = list(plant.keys()); rows=[]
    for _ in range(3000):
        e = RNG.choice(esp_list); txt = RNG.choice(plant[e]).replace("{t}",RNG.choice(tiempos))
        rows.append((txt,e,RNG.choice(["alta","media","baja"],p=prio[e]),RNG.choice(centros["centro_id"].values)))
    pd.DataFrame(rows,columns=["texto","especialidad","prioridad","centro_id"]).to_csv(os.path.join(carpeta,"notas_clinicas.csv"),index=False)

    # --- Wearable (señal) ---
    rows=[]
    for s in range(1,201):
        fcb=RNG.normal(66,6); pb=RNG.normal(7500,2500); sb=RNG.normal(7.0,0.8); anom=RNG.random()<0.05
        for dia in range(1,31):
            fc=fcb+RNG.normal(0,3); pasos=max(0,pb+RNG.normal(0,1500)); sue=float(np.clip(sb+RNG.normal(0,0.6),3,11))
            if anom and dia in (14,15,16): fc+=RNG.uniform(18,30); sue-=RNG.uniform(1.5,3)
            rows.append((f"S{s:03d}",dia,round(fc,1),int(pasos),round(sue,1)))
    pd.DataFrame(rows,columns=["sujeto_id","dia","fc_reposo","pasos","horas_sueno"]).to_csv(os.path.join(carpeta,"wearable.csv"),index=False)
    return carpeta

generar_datos_clinicos(".")
print("Datos sintéticos listos en la carpeta actual. (Recuerda: son datos SINTÉTICOS, no reales.)")

## 1. Conoce tu serie: cargar, ordenar y dibujar

Antes de nada, ¿por qué esto es distinto a todo lo que hemos hecho hasta ahora? En `pacientes.csv` cada fila era una persona; el orden de las filas era irrelevante. En `urgencias_diarias.csv` cada fila es **un día**, y el orden **es** la información: el ingreso de un lunes se entiende mejor si sabemos qué pasó el domingo y qué pasó el lunes anterior. A esto lo llamamos **serie temporal**: una secuencia de valores donde las observaciones **no son independientes**.

Empezamos por lo más básico y a la vez más importante: cargar la fecha *como fecha* (no como texto) y **ordenar** la tabla cronológicamente. Si esto se hace mal, todo lo que viene después se tuerce.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# parse_dates=["fecha"] convierte la columna de texto en una fecha real de Python/pandas
urg = pd.read_csv("urgencias_diarias.csv", parse_dates=["fecha"])
urg = urg.sort_values("fecha").reset_index(drop=True)   # ordenar por fecha es obligatorio en una serie

print("Filas y columnas:", urg.shape)
print("\nPrimer día de la serie:", urg["fecha"].min().date())
print("Último día de la serie:", urg["fecha"].max().date())
urg.head()

**Lo que vemos:** cada fila es un día, con sus `ingresos` en urgencias y tres columnas de contexto (`festivo`, `temporada_gripe`, `temperatura`). La serie cubre unos dos años completos, día a día, sin huecos. Fíjate en la columna `fecha`: gracias a `parse_dates`, pandas la trata como una fecha de verdad (podemos preguntarle "¿qué día de la semana es?"), no como un simple texto.


### 1.1 Un vistazo numérico antes de dibujar

Antes de la gráfica, unos números de referencia: ¿entre qué valores se mueven los ingresos diarios?


In [ ]:
urg["ingresos"].describe().round(1)

**Lo que vemos:** los ingresos diarios rondan, en promedio, un valor típico de referencia, pero con una dispersión notable entre el día más tranquilo y el más cargado. Esa variabilidad es justo lo que vamos a intentar explicar: ¿es puro azar, o hay un patrón detrás (el día de la semana, la época del año)? Vamos a la gráfica para empezar a verlo.


### 1.2 Dibujamos la serie completa

La forma más honesta de entender una serie es, sencillamente, **dibujarla**. Ponemos la fecha en el eje horizontal y los ingresos diarios en el vertical, y dejamos que el ojo haga el primer diagnóstico.


In [ ]:
plt.figure(figsize=(13, 4.5))
plt.plot(urg["fecha"], urg["ingresos"], color="#00AEC7", linewidth=0.9)
plt.xlabel("Fecha")
plt.ylabel("Ingresos diarios en urgencias")
plt.title("Ingresos diarios en urgencias — serie completa")
plt.tight_layout()
plt.show()

**Lo que vemos (y cómo se lee):** tres patrones saltan a la vista, aunque estén entremezclados en la misma línea:

- Un **serrucho** rápido y constante: la serie sube y baja dentro de cada semana. Es el **ritmo semanal** (más carga entre semana y fines de semana, como veremos en el bloque 2).
- Unas **jorobas** que se repiten cada año, con picos más altos en los meses de invierno y valles en verano. Es la **estacionalidad anual**, que aquí se ve reforzada por la temporada de gripe.
- Un poco de **ruido** día a día que no sigue ningún patrón fijo (un día con más accidentes, otro más tranquilo de lo esperado): la parte que ningún modelo razonable debería intentar memorizar.

Esta primera mirada ya nos dice qué información tendrá que capturar cualquier modelo que construyamos: el día de la semana, el mes del año, y el contexto (festivos, gripe).


### 1.3 Un acercamiento: dos meses cualquiera

La gráfica completa comprime dos años en una sola imagen y el serrucho semanal casi no se distingue. Hagamos zoom sobre un par de meses para verlo con nitidez.


In [ ]:
zoom = urg[(urg["fecha"] >= "2024-01-01") & (urg["fecha"] < "2024-03-01")]

plt.figure(figsize=(13, 4.5))
plt.plot(zoom["fecha"], zoom["ingresos"], "o-", color="#00AEC7", markersize=4)
plt.xlabel("Fecha")
plt.ylabel("Ingresos diarios en urgencias")
plt.title("Ingresos diarios — enero y febrero (acercamiento)")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

**Lo que vemos:** ahora sí se aprecia el vaivén semana a semana: picos y valles que se repiten con una cadencia de 7 días. Es exactamente el patrón que un profesional de urgencias reconocería de inmediato: hay días de la semana que cargan sistemáticamente más que otros. Vamos a cuantificarlo en el siguiente bloque.


## 2. Estacionalidad: ¿qué día y qué mes cargan más?

"Los lunes hay más ingresos" o "en invierno se colapsa más" son afirmaciones que cualquier profesional de urgencias haría sin pensarlo dos veces. Vamos a comprobarlas con datos, no con impresiones: calculamos el **promedio de ingresos por día de la semana** y por **mes del año**, y lo dibujamos en barras. Esto es precisamente **estacionalidad**: patrones que se repiten a intervalos fijos (cada 7 días, cada 12 meses).


### 2.1 Extraemos día de la semana y mes de la fecha

pandas puede leer directamente el día de la semana y el mes desde una columna de fecha, sin que tengamos que calcularlo a mano.


In [ ]:
urg["dia_semana"] = urg["fecha"].dt.dayofweek   # 0 = lunes, 1 = martes, ..., 6 = domingo
urg["mes"] = urg["fecha"].dt.month                # 1 = enero, ..., 12 = diciembre

nombres_dias = ["Lunes", "Martes", "Miércoles", "Jueves", "Viernes", "Sábado", "Domingo"]
urg["nombre_dia"] = urg["dia_semana"].map(lambda d: nombres_dias[d])

urg[["fecha", "nombre_dia", "mes", "ingresos"]].head(7)

**Lo que hemos hecho:** cada fila tiene ahora, además de la fecha, su **día de la semana** (0 a 6) y su **mes** (1 a 12), leídos directamente de esa fecha. Son las dos primeras piezas de calendario que necesitaremos tanto para el análisis de estacionalidad como, más adelante, para el propio modelo.


### 2.2 Media de ingresos por día de la semana

Agrupamos por día de la semana y calculamos la media de ingresos en cada uno.


In [ ]:
media_por_dia = urg.groupby("nombre_dia")["ingresos"].mean().reindex(nombres_dias)
media_por_dia.round(1)

In [ ]:
plt.figure(figsize=(8, 4.5))
plt.bar(media_por_dia.index, media_por_dia.values, color="#00AEC7")
plt.axhline(urg["ingresos"].mean(), color="#E4572E", linestyle="--", label="Media global")
plt.xlabel("Día de la semana")
plt.ylabel("Ingresos medios")
plt.title("Ingresos medios en urgencias por día de la semana")
plt.xticks(rotation=20)
plt.legend()
plt.tight_layout()
plt.show()

**Lo que vemos (lectura clínica):** el **lunes** despunta claramente por encima de los demás días entre semana — es el día en que se acumula todo lo que se ha ido aguantando durante el fin de semana, cuando la atención primaria estaba cerrada. El **sábado y el domingo** también quedan por encima del resto de días laborables, aunque no tanto como el lunes: sin consultas de primaria abiertas, urgencias es la única puerta de acceso. El valle se sitúa a mitad de semana (martes a jueves), cuando el sistema ya ha absorbido la demanda acumulada y la atención primaria funciona a pleno rendimiento. Este patrón semanal tan marcado es la primera pista de que el "día de la semana" tendrá que ser una variable de peso en cualquier modelo.


### 2.3 Media de ingresos por mes del año

Repetimos el mismo ejercicio, ahora agrupando por mes, para ver la estacionalidad **anual**.


In [ ]:
nombres_meses = ["Ene", "Feb", "Mar", "Abr", "May", "Jun", "Jul", "Ago", "Sep", "Oct", "Nov", "Dic"]
media_por_mes = urg.groupby("mes")["ingresos"].mean()
media_por_mes.index = [nombres_meses[m - 1] for m in media_por_mes.index]
media_por_mes.round(1)

In [ ]:
plt.figure(figsize=(9, 4.5))
plt.bar(media_por_mes.index, media_por_mes.values, color="#00AEC7")
plt.axhline(urg["ingresos"].mean(), color="#E4572E", linestyle="--", label="Media global")
plt.xlabel("Mes")
plt.ylabel("Ingresos medios")
plt.title("Ingresos medios en urgencias por mes del año")
plt.legend()
plt.tight_layout()
plt.show()

**Lo que vemos (lectura clínica):** los meses de **invierno** (diciembre, enero, febrero) se sitúan claramente por encima de la media, mientras que los meses de **verano** caen por debajo. No es casualidad: es la temporada alta de patología respiratoria, reforzada además por la **temporada de gripe**, que en nuestros datos coincide justamente con esos meses. Entre estos dos hallazgos — el pico de los lunes y la joroba invernal — ya tenemos identificados los dos patrones de estacionalidad más importantes de la serie: uno **semanal** y otro **anual**. Ambos deberán entrar, de una forma u otra, en el modelo que construyamos.


## 3. La lección crítica: validar sin dejar que el futuro se cuele

Llegamos al punto más importante de esta unidad, y conviene detenerse en él con calma. En unidades anteriores, para evaluar un modelo sobre `pacientes.csv`, separábamos los datos **al azar** en entrenamiento y test — y era lo correcto, porque cada paciente es una observación independiente: barajar las filas no destruye ninguna información.

Con una serie temporal, **eso es exactamente lo que no hay que hacer.**

**¿Por qué es una trampa?** Imagina que barajamos las fechas al azar y, por pura casualidad, un día de test —pongamos el 15 de febrero— queda con su día anterior (14 de febrero) y su día siguiente (16 de febrero) dentro del conjunto de **entrenamiento**. Al modelo le basta con **interpolar entre sus vecinos** para clavar casi cualquier predicción: no ha aprendido el patrón general, ha memorizado el entorno inmediato de ese día concreto. Peor todavía: si el test contiene días de un invierno y el entrenamiento incluye días *posteriores* de ese mismo invierno, el modelo habría "visto" cómo evolucionó la ola de gripe **después** de la fecha que le pedimos predecir. Es la versión temporal exacta de la fuga de datos que ya conocimos: usar información del futuro para predecir el pasado.

El resultado de este error es siempre el mismo guion: las métricas en la evaluación salen espectaculares, la ilusión crece… y en el primer mes de uso real el modelo falla, porque en la vida real **nadie te regala el dato del día siguiente**.


{% hint style="warning" %}
**⚠️ Aviso · Por qué un split aleatorio es tramposo aquí**

Un `train_test_split` al azar sobre una serie temporal reparte los días de entrenamiento y de test **mezclados en el tiempo**. Esto tiene dos efectos, ambos graves:

1. **Vecindad**: casi cada día de test tiene su día anterior y su día siguiente dentro del entrenamiento, así que el modelo puede "adivinar" por proximidad en vez de aprender el patrón real.
2. **Fuga de futuro**: el modelo puede entrenar con fechas posteriores a las que se le pide predecir, algo que en producción real es imposible — el futuro, sencillamente, no existe todavía cuando hay que decidir.

La regla que resuelve ambos problemas de un solo golpe: **entrenar solo con el pasado, predecir el futuro, nunca al revés.**
{% endhint %}


### 3.1 La solución: el corte temporal

La alternativa correcta es tan simple de aplicar como importante de entender: elegimos una **fecha de corte**, todo lo anterior a esa fecha es **entrenamiento** (el pasado, que el modelo puede estudiar) y todo lo posterior es **test** (el futuro, que el modelo tiene que adivinar sin haberlo visto nunca). Nada de barajar: la serie se corta en dos tramos consecutivos en el tiempo.

Usaremos un reparto habitual: el **80 % más antiguo** de los días para entrenar, y el **20 % más reciente** — el futuro real de la serie — para test.


In [ ]:
# Fecha de corte: el percentil 80 de las fechas de la serie
corte = urg["fecha"].quantile(0.8)
print("Fecha de corte:", corte.date())

# NUNCA barajar en el tiempo: todo lo anterior al corte es pasado (entreno), lo posterior es futuro (test)
train = urg[urg["fecha"] <= corte].copy()
test = urg[urg["fecha"] > corte].copy()

print(f"\nEntrenamiento (pasado):  {len(train)} días  ->  de {train['fecha'].min().date()} a {train['fecha'].max().date()}")
print(f"Test (futuro):           {len(test)} días  ->  de {test['fecha'].min().date()} a {test['fecha'].max().date()}")

**Lo que hemos hecho:** el conjunto de entrenamiento contiene el **80 % de días más antiguos** de la serie; el de test, el **20 % más reciente**, sin ningún solapamiento de fechas ni mezcla. El modelo que entrenemos a continuación solo podrá "ver" el tramo de entrenamiento; el tramo de test se queda guardado, intacto, para comprobar al final cómo se comporta frente a un futuro que nunca estudió. Así es exactamente como funcionaría en la vida real: hoy solo tenemos el pasado, y el mañana hay que predecirlo sin haberlo visto.


### 3.2 Lo vemos en la propia gráfica

Antes de seguir, conviene **ver** el corte sobre la serie, para que quede grabada la idea: una línea vertical separa limpiamente el pasado (entrenamiento) del futuro (test).


In [ ]:
plt.figure(figsize=(13, 4.5))
plt.plot(train["fecha"], train["ingresos"], color="#00AEC7", linewidth=0.9, label="Entrenamiento (pasado)")
plt.plot(test["fecha"], test["ingresos"], color="#E4572E", linewidth=0.9, label="Test (futuro)")
plt.axvline(corte, color="gray", linestyle="--", alpha=0.8)
plt.xlabel("Fecha")
plt.ylabel("Ingresos diarios en urgencias")
plt.title("Corte temporal: pasado (entrenamiento) vs. futuro (test)")
plt.legend()
plt.tight_layout()
plt.show()

**Lo que vemos:** el tramo azul (pasado) y el tramo naranja (futuro) no se solapan en ningún punto de la línea de tiempo. Esa separación limpia, sin mezcla, es la garantía de que cualquier resultado que obtengamos con este reparto es **honesto**: el modelo se examinará prediciendo días que jamás estudió, exactamente como tendría que hacerlo el día que se pusiera a trabajar de verdad en un hospital.


## 4. Un modelo de ML con "pistas" de calendario

Un modelo de árboles de decisión (como el que usaremos, `HistGradientBoostingRegressor`) no entiende "el tiempo" por sí mismo: no sabe que el 15 de enero viene después del 14 de enero. Lo que sí puede hacer es aprender relaciones entre **columnas**. Así que la estrategia es **convertir el calendario en columnas** — características (*features*) que el modelo pueda usar directamente:

- **`dow`** (día de la semana, 0–6): para capturar el pico de los lunes.
- **`mes`** (1–12): para capturar la estacionalidad anual.
- **`festivo`** (0/1): los festivos se comportan como pequeños domingos.
- **`temporada_gripe`** (0/1): la ola de invierno que se monta encima de la estacionalidad anual.
- **`temperatura`**: contexto adicional, relacionado con la propia estacionalidad.

Fíjate en algo importante: **todas estas columnas ya las conocemos de antemano** el día que hacemos la predicción — el día de la semana y el mes se derivan del propio calendario, y festivo, temporada de gripe y una previsión de temperatura son información disponible *antes* de que llegue el día. No hay fuga: ninguna de ellas usa datos "del futuro" que no tendríamos en la práctica.


### 4.1 Preparamos las features de calendario en train y test

Aplicamos la misma preparación a los dos tramos — importante: **por separado**, cada uno con sus propias fechas, sin volver a mezclarlos.


In [ ]:
features = ["dia_semana", "mes", "festivo", "temporada_gripe", "temperatura"]

X_train, y_train = train[features], train["ingresos"]
X_test, y_test = test[features], test["ingresos"]

print("Features utilizadas:", features)
X_train.head()

**Lo que hemos hecho:** cada fila de `X_train` y `X_test` es un día, descrito solo por sus cinco pistas de calendario. La columna que queremos predecir, `ingresos`, se guarda aparte en `y_train` e `y_test`. El modelo tendrá que aprender la relación entre estas cinco columnas y el número de ingresos, usando **solo** el tramo de entrenamiento.


### 4.2 Entrenamos el modelo

Usamos `HistGradientBoostingRegressor`, un modelo de *gradient boosting* — una familia de árboles de decisión que se van corrigiendo unos a otros — rápido y que funciona bien sin apenas ajuste manual. Lo entrenamos **únicamente** con el tramo de entrenamiento (el pasado).


In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor

modelo = HistGradientBoostingRegressor(random_state=0)
modelo.fit(X_train, y_train)

print("Modelo entrenado con", len(X_train), "días de historia (el tramo de entrenamiento).")

**Lo que hemos hecho:** el modelo ha aprendido, a partir del pasado, cómo el día de la semana, el mes, los festivos, la gripe y la temperatura se combinan para producir un determinado número de ingresos. Todavía no sabemos si lo ha aprendido bien — eso solo se puede juzgar mirando cómo predice el **futuro** que nunca vio, que es justo el siguiente paso.


### 4.3 Predecimos el test y calculamos el error (MAE)

Pedimos al modelo que prediga los ingresos del tramo de test (el futuro) usando solo sus columnas de calendario, y comparamos con lo que realmente ocurrió. Usamos el **MAE** (error absoluto medio): en promedio, cuántos ingresos de más o de menos se equivoca el modelo cada día. Es una métrica que se lee directamente en la misma unidad que el propio dato — personas — y por eso es tan cómoda de comunicar en clínica.


In [ ]:
from sklearn.metrics import mean_absolute_error

pred_modelo = modelo.predict(X_test)
mae_modelo = mean_absolute_error(y_test, pred_modelo)

print(f"MAE del modelo en test (futuro nunca visto): {mae_modelo:.1f} ingresos/día")

**Lo que significa:** en promedio, el modelo se equivoca por unos pocos ingresos al día al predecir el futuro — nada mal para un servicio que mueve una cifra de ingresos bastante mayor cada jornada. Y, sobre todo: esta cifra es **honesta**, porque se ha calculado sobre días que el modelo jamás estudió durante el entrenamiento. Antes de decidir si es un buen resultado o no, vamos a **verlo** en una gráfica y, en el bloque siguiente, a compararlo con un punto de referencia sencillo.


### 4.4 Dibujamos predicho vs. real en el test

La gráfica más honesta para un modelo de forecasting: superponer, día a día del tramo de test, lo que **predijo** el modelo y lo que **realmente ocurrió**.


In [ ]:
plt.figure(figsize=(13, 4.5))
plt.plot(test["fecha"], y_test.values, color="#00AEC7", linewidth=1.3, label="Real")
plt.plot(test["fecha"], pred_modelo, color="#E4572E", linewidth=1.3, linestyle="--", label="Predicho (modelo con calendario)")
plt.xlabel("Fecha")
plt.ylabel("Ingresos diarios en urgencias")
plt.title(f"Predicho vs. real en el test (futuro) — MAE = {mae_modelo:.1f}")
plt.legend()
plt.tight_layout()
plt.show()

**Lo que vemos:** la línea discontinua (predicción) sigue de cerca la forma general de la línea continua (realidad) — capta el vaivén semanal y el nivel general del tramo. No es una copia perfecta día a día (ahí está precisamente el "ruido" que decíamos que ningún modelo razonable debería perseguir), pero sí sigue bien la tendencia y la estacionalidad. Es exactamente el comportamiento que esperaríamos de un modelo que ha aprendido patrones de calendario reales, evaluado de forma honesta sobre un futuro que nunca vio.


## 5. ¿Aporta algo el modelo? Comparamos con un baseline ingenuo

Un MAE por sí solo no dice si el modelo es bueno: falta un punto de comparación. En series temporales, el listón mínimo que cualquier modelo tiene que superar es un **baseline ingenuo** — una regla tan simple que ni siquiera necesita aprender nada. Probamos dos:

- **Baseline "media"**: predecir todos los días de test con la **media histórica** de ingresos (el número más simple posible).
- **Baseline "hace 7 días"**: predecir cada día como **el valor real de hace exactamente una semana** — aprovechando, sin ningún modelo, el fuerte ritmo semanal que ya vimos en el bloque 2 ("el próximo lunes se parecerá al lunes pasado").

Si nuestro modelo de calendario no logra batir a estos dos rivales tan sencillos, todo el esfuerzo de entrenarlo no se justifica.


### 5.1 Baseline 1 — la media histórica

La predicción más simple imaginable: todos los días de test reciben el mismo valor, la media de ingresos observada en el tramo de entrenamiento.


In [ ]:
media_train = y_train.mean()
pred_media = np.full(len(y_test), media_train)

mae_media = mean_absolute_error(y_test, pred_media)
print(f"Media histórica de entrenamiento: {media_train:.1f} ingresos/día")
print(f"MAE del baseline 'media': {mae_media:.1f} ingresos/día")

**Lo que vemos:** al no distinguir ni días de la semana ni meses, este baseline se equivoca considerablemente más que el modelo de calendario del bloque 4. Tiene sentido: predecir siempre el mismo número, sin importar si es lunes o domingo, invierno o verano, ignora toda la estacionalidad que hemos visto que existe.


### 5.2 Baseline 2 — el valor de hace 7 días

Un baseline mucho más inteligente para una serie con ritmo semanal tan marcado: predecir el ingreso de cada día como el ingreso real registrado **una semana antes**. Para poder aplicarlo en el test necesitamos también los 7 últimos días del entrenamiento (los "hace 7 días" de los primeros días de test).


In [ ]:
# Concatenamos train + test ordenados por fecha para poder mirar "7 días atrás" también en los primeros días del test
serie_completa = pd.concat([train, test]).sort_values("fecha").reset_index(drop=True)
serie_completa["ingresos_hace_7d"] = serie_completa["ingresos"].shift(7)

# Nos quedamos solo con las filas de test, ya con su valor "de hace 7 días" adjunto
pred_hace_7d = serie_completa[serie_completa["fecha"] > corte]["ingresos_hace_7d"].values

mae_hace_7d = mean_absolute_error(y_test, pred_hace_7d)
print(f"MAE del baseline 'como hace 7 días': {mae_hace_7d:.1f} ingresos/día")

**Lo que vemos:** este segundo baseline, aunque sigue siendo una regla muy simple ("este lunes será como el lunes pasado"), mejora bastante al de la media — precisamente porque, sin llamarlo así, **incorpora el patrón semanal** que sabemos que existe. Es la prueba de que en esta serie el ritmo de 7 días es tan fuerte que hasta una regla ingenua que lo aprovecha se vuelve un rival serio.


### 5.3 La comparación final: tabla y gráfica conjunta

Reunimos los tres MAE en una tabla y dibujamos los tres pronósticos junto a la realidad, para decidir con criterio cuál merece la pena.


In [ ]:
comparacion = pd.DataFrame({
    "Método": ["Baseline: media histórica", "Baseline: como hace 7 días", "Modelo ML con calendario"],
    "MAE (ingresos/día)": [mae_media, mae_hace_7d, mae_modelo],
}).sort_values("MAE (ingresos/día)").reset_index(drop=True)

comparacion.round(1)

In [ ]:
plt.figure(figsize=(13, 4.5))
plt.plot(test["fecha"], y_test.values, color="black", linewidth=1.5, label="Real")
plt.plot(test["fecha"], pred_modelo, color="#E4572E", linewidth=1.1, linestyle="--", label=f"Modelo ML (MAE={mae_modelo:.1f})")
plt.plot(test["fecha"], pred_hace_7d, color="#00AEC7", linewidth=1.1, linestyle=":", label=f"Hace 7 días (MAE={mae_hace_7d:.1f})")
plt.axhline(media_train, color="gray", linewidth=1.1, linestyle="-.", label=f"Media histórica (MAE={mae_media:.1f})")
plt.xlabel("Fecha")
plt.ylabel("Ingresos diarios en urgencias")
plt.title("Los tres métodos frente a la realidad, en el mismo test (futuro)")
plt.legend()
plt.tight_layout()
plt.show()

**Lo que vemos y lo que decidimos:**

- La **media histórica** (línea horizontal gris) es, con diferencia, el peor de los tres: es sorda a todo lo que hemos aprendido sobre estacionalidad semanal y anual.
- El baseline **"como hace 7 días"** ya es un rival digno — recoge el ritmo semanal sin necesitar entrenar nada —, pero no distingue meses, ni festivos, ni temporada de gripe: no puede anticipar, por ejemplo, que un festivo concreto se comportará como domingo.
- El **modelo de ML con calendario** consigue el MAE más bajo de los tres: además del ritmo semanal, incorpora la estacionalidad anual, los festivos, la gripe y la temperatura, todo a la vez.

**La lección que nos llevamos no es solo "el modelo gana"**: es que un baseline ingenuo bien elegido (aquí, "como hace 7 días") ya captura buena parte del patrón más fuerte de la serie, y es el listón real que cualquier modelo tiene que justificar superar. Si el modelo de calendario apenas mejorase sobre ese baseline, la conclusión honesta sería quedarse con la regla simple — más fácil de explicar, de mantener y de confiar. Aquí sí aporta una mejora clara, y por eso se gana el puesto.


### 5.4 Y las series por paciente, ¿qué cambia?

Todo lo aprendido en esta unidad —el orden como información, la estacionalidad, la validación temporal sin fugas— se aplica exactamente igual si cambiamos de escala: del hospital entero al paciente individual. `wearable.csv`, otro de nuestros datasets sintéticos, contiene series **diarias por sujeto** (frecuencia cardiaca en reposo, pasos, horas de sueño). Una frecuencia cardiaca en reposo que sube de forma sostenida durante varios días, por ejemplo, puede preceder a una descompensación o a una infección — y para detectarlo con honestidad harían falta las mismas reglas: nunca entrenar con datos posteriores al día que se quiere vigilar, y desconfiar de cualquier variable que en la práctica no estuviera disponible todavía ese día. Cambia la escala, no los principios.


## 6. Qué hemos aprendido

- **El orden es información.** En una serie temporal, a diferencia de una tabla de pacientes, no se puede barajar: los ingresos de hoy dependen de ayer, del lunes pasado y del invierno anterior.
- **Dos estacionalidades muy claras** en `urgencias_diarias.csv`: la **semanal** (pico el lunes, elevado también el fin de semana) y la **anual** (invierno muy por encima de verano, reforzado por la temporada de gripe).
- **La lección crítica: validación temporal sin fugas.** Un `train_test_split` al azar sobre una serie mezcla pasado y futuro y produce una fuga de datos devastadora — el modelo "interpola" entre vecinos o incluso "ve" cómo evolucionó lo que se le pide predecir. La solución es el **corte temporal**: todo lo anterior a una fecha es entrenamiento, todo lo posterior es test, sin solapamiento ni mezcla.
- **Un modelo de ML puede usar el calendario como pista**: día de la semana, mes, festivo, temporada de gripe y temperatura, todas ellas disponibles de antemano el día de la predicción — sin fuga. Evaluado de forma honesta sobre el futuro, su MAE fue el más bajo de los tres métodos comparados.
- **Comparar siempre contra un baseline ingenuo.** La media histórica es un rival débil; "como hace 7 días" ya es un rival serio en una serie tan semanal como esta. Un modelo solo se gana el puesto si mejora de forma clara sobre el listón más simple que ya funciona razonablemente bien.
- **La misma lógica se aplica a escala de paciente** (`wearable.csv`): el tiempo importa igual en la frecuencia cardiaca de una persona que en los ingresos de un hospital entero.

**Puente a la U8:** hasta aquí, todo el aprendizaje automático ha trabajado sobre tablas y series — datos con filas y columnas, por muy bien que hayamos aprendido a prepararlas. En la siguiente unidad entramos en la familia que ha protagonizado la última década y que en medicina brilla justo donde la tabla no llega — la imagen, el texto y la señal —: las redes neuronales y el *deep learning*.


## 7. Y esto, ¿cómo se lo pido a un asistente de IA?

Recuerda el principio del curso: **tú pones el criterio, la IA escribe el código.** En series temporales, el criterio más valioso que puedes aportar es precisamente el que hemos entrenado en esta unidad: saber detectar y rechazar una validación tramposa. Un buen encargo para reproducir y ampliar este cuaderno sería:

> *"Actúa como científico de datos clínicos senior especializado en forecasting. Con `urgencias_diarias.csv` (datos **sintéticos**: fecha, ingresos, festivo, temporada_gripe, temperatura), en español y por celdas: (1) Carga con `parse_dates`, ordena por fecha y dibuja la serie completa, interpretando qué patrones se ven. (2) Calcula y dibuja en barras la media de ingresos por día de la semana y por mes; interpreta el pico de los lunes y la carga invernal. (3) Antes de modelar, **explícame por qué un `train_test_split` aleatorio sería una trampa aquí** y haz en su lugar un **corte temporal** (80 % pasado para entrenar, 20 % futuro para test, sin barajar). (4) Entrena un `HistGradientBoostingRegressor` con features de calendario (día de semana, mes, festivo, temporada_gripe, temperatura), calcula el MAE en el test y dibuja predicho vs. real. (5) Compárame el resultado con un baseline de la media histórica y con un baseline 'como hace 7 días', en una tabla y una gráfica conjunta, y dime con criterio si el modelo se gana el puesto. Al terminar, dime qué patrón capta cada método y cuál usarías para planificar turnos de personal."*

Tu trabajo con su respuesta es el de siempre, pero aquí con una prioridad clara: comprobar **primero** que la partición sea temporal y no aleatoria — si el asistente propone barajar los datos o usar `train_test_split` sin más, recházalo sin mirar el resto: es el error clásico de las series temporales, y detectarlo es exactamente el criterio que este curso quiere darte. Después, comprueba que compare contra un baseline ingenuo antes de declarar ganador a ningún modelo.
